# pAIr 총괄 시험 (RunPod)

**이 노트북 하나만 JupyterLab 에 끌어다 놓고 `Run All` 하면 됩니다.** 코드·모델(약 700MB)은 구글 드라이브에서 받아 검증하고, 아래 시험을 차례로 돌린 뒤 **결과 zip 하나**를 만듭니다. 한 단계가 실패해도 다음 단계는 계속 돌고, 맨 끝에 단계별 성공/실패 요약이 나옵니다.

| 단계 | 내용 |
|---|---|
| ① | zip 받기 → 크기·sha256 검증 → 풀기 |
| ② | 환경 준비(apt 라이브러리, 가상환경, torch 2.6 미만이면 설치) |
| ③ | GPU·모델 sha256 확인 |
| ④ | pytest 전체 |
| ⑤ | 모델 속도 비교(compare) |
| ⑥ | 용량 측정(capacity) |
| ⑦ | 정확도 평가(정답 labels.csv 가 번들에 있을 때만) |
| ⑧ | 방 스캔 API `/v1/scan` (실제 모델 서버를 띄워 호출) |
| ⑨ | 스트림·콜백 재생(replay + sink) |
| ⑩ | 결과 zip 만들기 + `팀원4_결과요약.md`(zip 에 포함, 마지막 칸에 출력) |

사진은 zip 안 `photos/` → `/workspace/photos`(+`extra/`, `labels.csv`) 순서로 찾고, 둘 다 없으면 합성 사진을 씁니다.

끝나면 **파일 목록에서 `results_*.zip` 을 내려받고, Pod 를 반드시 정지(Stop)하거나 삭제(Terminate)** 하세요(켜 두면 계속 과금됩니다).

In [ ]:
# ===== 설정 (여기만 바꾸면 됩니다) =====
DRIVE_URL = ""  # 구글 드라이브 공유 링크('링크가 있는 모든 사용자'). 비워 두면 실행할 때 물어봅니다
ZIP_BYTES = 0  # make_bundle.py 가 채웁니다
ZIP_SHA256 = ""  # make_bundle.py 가 채웁니다
ZIP_NAME = "pair-all-tests.zip"

QUICK = True  # True: 짧게(각 시험 1조합·몇 분). False: 전체(용량 측정만 1~2시간)
ALLOW_CPU = False  # GPU 가 없어도 CPU 로 돌린다(PC 에서 노트북 확인할 때만 True)

RUN_ENV = True  # ② 환경 준비
RUN_PYTEST = True  # ④
RUN_COMPARE = True  # ⑤
RUN_CAPACITY = True  # ⑥
RUN_EVAL = True  # ⑦ (labels.csv 가 없으면 건너뜀)
RUN_SCAN = True  # ⑧
RUN_REPLAY = True  # ⑨

# ===== 단계 실행기: 실패해도 다음 단계로 넘어갑니다 =====
import hashlib
import os
import sys
import time
import traceback
import zipfile
from pathlib import Path

BASE = Path(os.environ.get("PAIR_HOME") or Path.home())  # 코드·가상환경·결과를 둘 곳(/workspace 권한 문제를 피해 홈 아래)
ROOT = BASE / "pair-all-tests"
RESULTS = []  # (이름, 상태, 초, 설명)


class Skip(Exception):
    pass


def step(name, enabled, fn):
    if not enabled:
        RESULTS.append((name, "건너뜀", 0.0, "스위치가 꺼져 있음"))
        print(f"\n===== {name}: 건너뜀(스위치 꺼짐) =====")
        return
    print(f"\n===== {name} 시작 =====", flush=True)
    t0 = time.time()
    try:
        detail = fn()
        RESULTS.append((name, "성공", time.time() - t0, str(detail or "")))
    except Exception as err:  # Skip 도 포함: 이름으로 구분
        status = "건너뜀" if type(err).__name__ == "Skip" else "실패"
        if status == "실패":
            traceback.print_exc()
        RESULTS.append((name, status, time.time() - t0, str(err)[:400]))
    print(f"===== {name}: {RESULTS[-1][1]} ({RESULTS[-1][2]:.0f}초) {RESULTS[-1][3][:200]}", flush=True)

In [ ]:
# ① zip 받기 -> 검증 -> 풀기
def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_zip():
    for folder in (Path.cwd(), BASE, Path("/workspace"), Path.cwd().parent):
        candidate = folder / ZIP_NAME
        if candidate.is_file():
            return candidate
    return None


def step_download():
    stamp = ROOT / ".extracted-sha256"
    if (ROOT / "src" / "pair_vision").is_dir() and stamp.is_file() and stamp.read_text().strip() == ZIP_SHA256:
        return f"이미 풀려 있음: {ROOT}"
    zip_path = find_zip()
    if zip_path is None:
        url = DRIVE_URL if DRIVE_URL.startswith("http") else input("구글 드라이브 공유 링크를 붙여넣고 Enter: ").strip()
        os.system(f'"{sys.executable}" -m pip install -q gdown')
        import gdown

        zip_path = BASE / ZIP_NAME
        print("받는 중 (약 700MB, 몇 분)...")
        gdown.download(url, str(zip_path), quiet=False, fuzzy=True)
        if not zip_path.is_file():
            raise RuntimeError("받지 못했습니다. 링크가 '링크가 있는 모든 사용자'로 공유돼 있는지 확인하세요.")
    size = zip_path.stat().st_size
    if ZIP_BYTES and size != ZIP_BYTES:
        raise RuntimeError(f"크기가 다릅니다({size:,} != {ZIP_BYTES:,}). {zip_path} 를 지우고 다시 받으세요.")
    if ZIP_SHA256 and sha256_of(zip_path) != ZIP_SHA256:
        raise RuntimeError(f"sha256 이 다릅니다. {zip_path} 를 지우고 다시 받으세요.")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(BASE)
    stamp.write_text(ZIP_SHA256)
    return f"{zip_path.name} {size / 1e6:.0f}MB 검증·압축 해제 -> {ROOT}"


step("① 받기·검증·풀기", True, step_download)
if (ROOT / "src").is_dir():
    sys.path.insert(0, str(ROOT))
    from tools.runpod import all_tests as T

    Skip = T.Skip  # 단계 함수가 던지는 Skip 과 같은 이름으로 취급
    ctx = T.Ctx(ROOT, quick=QUICK, allow_cpu=ALLOW_CPU)
else:
    print("① 이 실패하면 아래 시험은 모두 건너뜁니다.")
    T = ctx = None


def run(name, enabled, fn_name):
    if T is None:
        RESULTS.append((name, "건너뜀", 0.0, "① 실패"))
        return
    step(name, enabled, lambda: getattr(T, fn_name)(ctx))

In [ ]:
run("② 환경 준비", RUN_ENV, "step_env")

In [ ]:
run("③ GPU·모델 확인", True, "step_gpu_models")

In [ ]:
run("④ pytest 전체", RUN_PYTEST, "step_pytest")

In [ ]:
run("⑤ 모델 속도 비교", RUN_COMPARE, "step_compare")

In [ ]:
run("⑥ 용량 측정", RUN_CAPACITY, "step_capacity")

In [ ]:
run("⑦ 정확도 평가", RUN_EVAL, "step_eval")

In [ ]:
run("⑧ 방 스캔 API", RUN_SCAN, "step_scan")

In [ ]:
run("⑨ 스트림·콜백 재생", RUN_REPLAY, "step_replay")

In [ ]:
# ⑩ 요약 + 결과 zip
lines = ["# 총괄 시험 요약", "", "| 단계 | 결과 | 시간(초) | 설명 |", "|---|---|---|---|"]
for name, status, secs, detail in RESULTS:
    lines.append(f"| {name} | {status} | {secs:.0f} | {detail.replace('|', '/')[:300]} |")
summary = "\n".join(lines) + "\n"
print(summary)
team_text = ""
if T is not None:
    try:
        from tools.runpod import team_summary

        team_text = team_summary.write(ROOT, ctx.results, RESULTS)  # 팀원4_결과요약.md (결과 zip 에 들어감)
    except Exception:
        traceback.print_exc()
    out = T.pack_results(ctx, summary)
    print("결과 zip:", out)
    copy = Path.cwd() / out.name  # JupyterLab 파일 목록에서 바로 보이도록 현재 폴더에도 둔다
    if copy.resolve() != out.resolve():
        import shutil

        shutil.copy2(out, copy)
        print("복사본:", copy)
    try:
        from IPython.display import FileLink, display

        display(FileLink(str(copy if copy.exists() else out)))
    except Exception:
        pass
if team_text:
    print("\n" + "=" * 20 + " 팀원4_결과요약.md " + "=" * 20 + "\n")
    print(team_text)
failed = [r[0] for r in RESULTS if r[1] == "실패"]
print("\n실패한 단계:", failed or "없음")
print("끝났습니다. 위 zip(results_*.zip)을 내려받고, Pod 를 정지(Stop)하거나 삭제(Terminate)하세요.")